# KV-only ingest to PGKV (Colab)
Notebook này chỉ backfill `text_chunks` và `full_docs` từ `custom_kg_full.json` vào LightRAG PGKVStorage, không đụng Milvus/Neo4j.

In [ ]:
!pip -q install lightrag-hku asyncpg numpy

In [ ]:
import asyncio
import json
import os
from pathlib import Path

import numpy as np
from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc

## 1) Set Postgres env
Điền đúng thông tin Postgres dùng cho PGKVStorage.

In [ ]:
os.environ["POSTGRES_HOST"] = "localhost"
os.environ["POSTGRES_PORT"] = "5432"
os.environ["POSTGRES_USER"] = "admin"
os.environ["POSTGRES_PASSWORD"] = "supersecretpassword"
os.environ["POSTGRES_DATABASE"] = "legal_ai_audit"

# Workspace phải trùng với runtime backend
os.environ["POSTGRES_WORKSPACE"] = "legal_ai_platform"

## 2) Upload custom_kg_full.json

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

In [ ]:
def _batched_items(data: dict, batch_size: int):
    items = list(data.items())
    for i in range(0, len(items), batch_size):
        yield dict(items[i : i + batch_size])


def _build_chunks_dict(custom_kg: dict) -> dict:
    chunks_dict = {}
    for chunk in custom_kg.get("chunks", []):
        source_id = chunk.get("source_id")
        if not source_id:
            continue

        file_path = chunk.get("file_path", "unknown_source")
        chunks_dict[source_id] = {
            "content": chunk.get("content", ""),
            "source_id": source_id,
            "file_path": file_path,
            "full_doc_id": chunk.get("full_doc_id", file_path),
            "chunk_order_index": chunk.get("chunk_order_index", 0),
        }
    return chunks_dict


def _build_full_docs_dict(chunks_dict: dict) -> dict:
    grouped = {}
    for chunk in chunks_dict.values():
        doc_id = chunk.get("full_doc_id") or chunk.get("file_path") or "unknown_doc"
        grouped.setdefault(doc_id, []).append(chunk)

    full_docs = {}
    for doc_id, doc_chunks in grouped.items():
        doc_chunks_sorted = sorted(doc_chunks, key=lambda c: c.get("chunk_order_index", 0))
        merged_content = "\n\n".join(c.get("content", "") for c in doc_chunks_sorted if c.get("content"))
        full_docs[doc_id] = {
            "content": merged_content,
            "file_path": doc_id,
        }
    return full_docs


async def _dummy_embed(texts: list[str], **kwargs) -> np.ndarray:
    return np.zeros((len(texts), 8), dtype=np.float32)


async def _dummy_llm(prompt: str, system_prompt: str | None = None, history_messages=None, **kwargs):
    return ""

In [ ]:
async def backfill_kv_only(
    json_path: str = "/content/custom_kg_full.json",
    workspace: str = "legal_ai_platform",
    batch_size: int = 1000,
    skip_full_docs: bool = False,
):
    os.environ["POSTGRES_WORKSPACE"] = workspace

    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"JSON not found: {json_path}")

    with path.open("r", encoding="utf-8") as f:
        custom_kg = json.load(f)

    chunks_dict = _build_chunks_dict(custom_kg)
    if not chunks_dict:
        raise ValueError("No chunks found in JSON input.")

    full_docs_dict = _build_full_docs_dict(chunks_dict)

    work_dir = Path("/content/target_kv_only")
    work_dir.mkdir(parents=True, exist_ok=True)

    rag = LightRAG(
        working_dir=str(work_dir),
        llm_model_func=_dummy_llm,
        embedding_func=EmbeddingFunc(
            embedding_dim=8,
            max_token_size=64,
            func=_dummy_embed,
        ),
        kv_storage="PGKVStorage",
        vector_storage="NanoVectorDBStorage",
        graph_storage="NetworkXStorage",
    )

    await rag.initialize_storages()

    print(f"[KV] Upserting text_chunks: {len(chunks_dict)}")
    for batch in _batched_items(chunks_dict, batch_size):
        await rag.text_chunks.upsert(batch)

    if not skip_full_docs:
        print(f"[KV] Upserting full_docs: {len(full_docs_dict)}")
        for batch in _batched_items(full_docs_dict, batch_size):
            await rag.full_docs.upsert(batch)

    print("Done. PGKV backfill completed.")

In [ ]:
await backfill_kv_only(
    json_path="/content/custom_kg_full.json",
    workspace="legal_ai_platform",
    batch_size=1000,
    skip_full_docs=False,
)